# Bayesian Modeling of Cross-Country TB Treatment Success

## A Fully Bayesian MCMC Analysis of WHO Data, 2012–2023

> **Course:** Fundamentals of Statistical Learning II — M.Sc. in Data Science, a.y. 2025–2026

**Architecture:**
- **Code:** Stored in GitHub — edit locally, push to GitHub
- **Data/Runs/Outputs:** Stored in Google Drive — persists across Colab sessions
- **Runtime:** R (via IRkernel in Colab)

---
## Section A — Setup (Drive + Repo + R Packages)

In [ ]:
# A0) Mount Google Drive
# In R-runtime Colab, Drive is mounted via system() call
# Note: Colab R notebooks auto-mount Drive at /content/drive when you
# click "Mount Drive" in the sidebar, or you can do it programmatically:

if (!dir.exists("/content/drive")) {
  # Colab-specific: mount Google Drive
  system("echo 'Please mount Google Drive from the sidebar (folder icon → Mount Drive)'")
} else {
  cat("✅ Google Drive already mounted at /content/drive\n")
}

In [ ]:
# A1) Define paths: code in Colab VM; data/runs in Drive

GDRIVE_ROOT  <- "/content/drive/MyDrive/Projects"
PROJECT_NAME <- "FSL_2_Final_Project"
PROJECT_DRIVE <- file.path(GDRIVE_ROOT, PROJECT_NAME)

# Storage directories in Drive
DATA_DIR   <- file.path(PROJECT_DRIVE, "data")
DATA_RAW   <- file.path(DATA_DIR, "data_raw")
DATA_PROC  <- file.path(DATA_DIR, "data_processed")
RUNS_DIR   <- file.path(PROJECT_DRIVE, "runs")
OUTPUT_DIR <- file.path(PROJECT_DRIVE, "output")

# Ensure Drive skeleton
for (p in c(DATA_RAW, DATA_PROC, RUNS_DIR, OUTPUT_DIR)) {
  dir.create(p, recursive = TRUE, showWarnings = FALSE)
}

cat("📁 DATA_RAW: ", DATA_RAW, "\n")
cat("📁 DATA_PROC:", DATA_PROC, "\n")
cat("📁 RUNS_DIR: ", RUNS_DIR, "\n")
cat("📁 OUTPUT_DIR:", OUTPUT_DIR, "\n")

In [ ]:
# A2) Clone/Update code from GitHub → /content/

REPO_OWNER <- "armanfeili"
REPO_NAME  <- "FSL_2_Final_Project"
REPO_URL   <- paste0("https://github.com/", REPO_OWNER, "/", REPO_NAME, ".git")
REPO_PATH  <- file.path("/content", REPO_NAME)

if (dir.exists(REPO_PATH)) {
  cat(REPO_PATH, "exists → pulling latest...\n")
  system2("git", c("-C", REPO_PATH, "fetch", "--prune"))
  system2("git", c("-C", REPO_PATH, "checkout", "main"))
  system2("git", c("-C", REPO_PATH, "pull", "--ff-only"))
} else {
  cat("Cloning", REPO_URL, "→", REPO_PATH, "...\n")
  system2("git", c("clone", "--depth=1", REPO_URL, REPO_PATH))
}

cat("✅ Repository ready at:", REPO_PATH, "\n")

In [ ]:
# A3) Install and load R packages
# Colab R runtime comes with many packages pre-installed.
# We install any missing ones here.

required_packages <- c(
  # --- MCMC & Bayesian ---
  "rjags",        # Interface to JAGS
  "coda",         # MCMC diagnostics
  "MCMCvis",      # MCMC visualization
  
  # --- Data wrangling ---
  "tidyverse",    # dplyr, ggplot2, tidyr, readr, etc.
  "data.table",   # Fast data manipulation
  
  # --- Visualization ---
  "ggplot2",      # Plotting
  "patchwork",    # Combine plots
  "corrplot",     # Correlation matrices
  
  # --- Frequentist comparison ---
  "lme4",         # Mixed-effects models
  "VGAM",         # Beta-binomial regression
  
  # --- Utilities ---
  "yaml",         # Config files
  "knitr",        # Tables
  "car"           # VIF
)

# Install missing packages
installed <- rownames(installed.packages())
to_install <- setdiff(required_packages, installed)

if (length(to_install) > 0) {
  cat("Installing:", paste(to_install, collapse = ", "), "\n")
  install.packages(to_install, repos = "https://cloud.r-project.org", quiet = TRUE)
} else {
  cat("All packages already installed.\n")
}

# Install JAGS itself (system dependency)
if (system("which jags", ignore.stdout = TRUE, ignore.stderr = TRUE) != 0) {
  cat("Installing JAGS system library...\n")
  system("apt-get update -qq && apt-get install -y -qq jags", ignore.stdout = TRUE)
}

cat("✅ All packages installed\n")

In [ ]:
# A4) Load all libraries

suppressPackageStartupMessages({
  library(rjags)
  library(coda)
  library(MCMCvis)
  library(tidyverse)
  library(data.table)
  library(patchwork)
  library(corrplot)
  library(lme4)
  library(VGAM)
  library(yaml)
  library(knitr)
  library(car)
})

cat("✅ Libraries loaded\n")
cat("   R version:", R.version.string, "\n")
cat("   JAGS:     ", system("jags --version 2>&1 | head -1", intern = TRUE), "\n")

In [ ]:
# A5) Set seed and create run configuration

SEED <- 42
set.seed(SEED)

# ggplot2 theme
theme_set(theme_minimal(base_size = 13))

cat("✅ Seed set to", SEED, "\n")
cat("✅ ggplot2 theme set\n")

---
## Section B — Data Loading & Cleaning

In [ ]:
# B0) Load raw CSV data
# The data files should be in data/data_raw/ (either in the repo or Drive)

# Try repo path first, then Drive path
data_source <- if (dir.exists(file.path(REPO_PATH, "data", "data_raw"))) {
  file.path(REPO_PATH, "data", "data_raw")
} else {
  DATA_RAW
}

cat("Loading data from:", data_source, "\n")
cat("Files found:\n")
print(list.files(data_source, pattern = "\\.csv$"))

# Load the three key files
# (Update filenames to match your actual CSV names)
# outcomes  <- read_csv(file.path(data_source, "TB_outcomes_2026-04-04.csv"),     show_col_types = FALSE)
# burden    <- read_csv(file.path(data_source, "TB_burden_countries_2026-04-04.csv"), show_col_types = FALSE)
# data_dict <- read_csv(file.path(data_source, "TB_data_dictionary_2026-04-04.csv"), show_col_types = FALSE)

# cat("\n✅ Data loaded:\n")
# cat("   outcomes:", nrow(outcomes), "rows ×", ncol(outcomes), "cols\n")
# cat("   burden:  ", nrow(burden), "rows ×", ncol(burden), "cols\n")

In [ ]:
# B1) Data Cleaning Pipeline (per PROJECT_PLAN.md §6)
#
# Steps:
# 1. Keep needed columns
# 2. Construct success = newrel_succ, cohort = newrel_coh
# 3. Restrict to 2012–2023
# 4. Apply inclusion flag: rel_with_new_flg == 1
# 5. Drop invalid rows
# 6. Merge outcomes + burden by (iso3, year)
# 7. Standardize continuous predictors
# 8. Encode g_whoregion as factor
# 9. Lock main-analysis table

# TODO: Implement cleaning pipeline
cat("⏳ Data cleaning pipeline — implement per PROJECT_PLAN.md §6\n")

In [ ]:
# B2) Sample Attrition Table

# TODO: Fill with actual counts after cleaning
# attrition <- tribble(
#   ~Stage,                                ~Rows, ~Countries, ~Years,
#   "Raw merged rows",                       NA,       NA,      NA,
#   "After year restriction (2012–2023)",     NA,       NA,      NA,
#   "After comparability flag",               NA,       NA,      NA,
#   "After missingness removal",              NA,       NA,      NA,
#   "After cohort filter (≥50)",              NA,       NA,      NA
# )
# kable(attrition)

cat("⏳ Attrition table — fill after implementing cleaning\n")

---
## Section C — Exploratory Data Analysis

In [ ]:
# C0) EDA — Descriptive summaries (per PROJECT_PLAN.md §7)
#
# TODO:
# 1. Sample size summary
# 2. Cohort distribution histogram
# 3. Success rate distribution
# 4. Temporal trends by WHO region
# 5. Bivariate relationships
# 6. Correlation matrix & VIF screening
# 7. Country-level spread

cat("⏳ EDA section — implement per PROJECT_PLAN.md §7\n")

---
## Section D — MCMC Modeling with JAGS

In [ ]:
# D0) MCMC Run Configuration

RUN_ID <- paste0(format(Sys.time(), "%Y-%m-%d_%H-%M"), "_bayesian_tb")
RUN_DIR <- file.path(RUNS_DIR, RUN_ID)

for (subdir in c("mcmc_output", "plots", "diagnostics", "tables")) {
  dir.create(file.path(RUN_DIR, subdir), recursive = TRUE, showWarnings = FALSE)
}

# MCMC settings
mcmc_cfg <- list(
  n_chains   = 4,
  n_burnin   = 4000,
  n_iter     = 8000,
  n_thin     = 1,
  seed       = SEED
)

# Save config
write_yaml(mcmc_cfg, file.path(RUN_DIR, "mcmc_config.yaml"))

cat("🏷️ RUN_ID: ", RUN_ID, "\n")
cat("📁 RUN_DIR:", RUN_DIR, "\n")
cat("✅ MCMC config saved\n")

In [ ]:
# D1) Model 1 — Binomial Logistic Regression (Baseline)

model1_string <- "
model {
  for (i in 1:N) {
    logit(p[i]) <- beta0 + inprod(X[i,], beta[]) + gamma[region[i]]
    Y[i] ~ dbin(p[i], n[i])
  }

  # Priors
  beta0 ~ dnorm(0, 1/(2.5*2.5))
  for (j in 1:p) {
    beta[j] ~ dnorm(0, 1/(2.5*2.5))
  }
  for (r in 2:R) {
    gamma[r] ~ dnorm(0, 1/(2.5*2.5))
  }
  gamma[1] <- 0  # baseline region
}
"

cat("✅ Model 1 (Binomial Logistic) defined\n")

# TODO: Prepare data list, compile & run JAGS model
# jags_data <- list(Y = ..., n = ..., X = ..., region = ..., N = ..., p = ..., R = ...)
# model1 <- jags.model(textConnection(model1_string), data = jags_data,
#                       n.chains = mcmc_cfg$n_chains, n.adapt = 1000)
# update(model1, mcmc_cfg$n_burnin)
# samples1 <- coda.samples(model1, variable.names = c("beta0", "beta", "gamma"),
#                           n.iter = mcmc_cfg$n_iter, thin = mcmc_cfg$n_thin)

In [ ]:
# D2) Model 2 — Beta-Binomial Regression

model2_string <- "
model {
  for (i in 1:N) {
    # Logit link for mean
    logit(mu[i]) <- beta0 + inprod(X[i,], beta[]) + gamma[region[i]]

    # Beta-distributed latent probability
    alpha[i] <- mu[i] * phi
    betap[i] <- (1 - mu[i]) * phi
    theta[i] ~ dbeta(alpha[i], betap[i])

    # Observed successes
    Y[i] ~ dbin(theta[i], n[i])
  }

  # Priors
  beta0 ~ dnorm(0, 1/(2.5*2.5))
  for (j in 1:p) {
    beta[j] ~ dnorm(0, 1/(2.5*2.5))
  }
  for (r in 2:R) {
    gamma[r] ~ dnorm(0, 1/(2.5*2.5))
  }
  gamma[1] <- 0

  phi ~ dgamma(2, 0.1)
}
"

cat("✅ Model 2 (Beta-Binomial) defined\n")

# TODO: Compile & run JAGS model
# model2 <- jags.model(textConnection(model2_string), data = jags_data,
#                       n.chains = mcmc_cfg$n_chains, n.adapt = 1000)
# update(model2, mcmc_cfg$n_burnin)
# samples2 <- coda.samples(model2,
#   variable.names = c("beta0", "beta", "gamma", "phi"),
#   n.iter = mcmc_cfg$n_iter, thin = mcmc_cfg$n_thin)

In [ ]:
# D3) Model 3 — Hierarchical Beta-Binomial Regression

model3_string <- "
model {
  for (i in 1:N) {
    logit(mu[i]) <- beta0 + inprod(X[i,], beta[]) + gamma[region[i]] + u[country[i]]

    alpha[i] <- mu[i] * phi
    betap[i] <- (1 - mu[i]) * phi
    theta[i] ~ dbeta(alpha[i], betap[i])

    Y[i] ~ dbin(theta[i], n[i])
  }

  # Country random effects
  for (c in 1:C) {
    u[c] ~ dnorm(0, tau_u)
  }
  tau_u <- 1 / (sigma_u * sigma_u)
  sigma_u ~ dnorm(0, 1) T(0, )  # Half-Normal(0,1)

  # Fixed-effect priors
  beta0 ~ dnorm(0, 1/(2.5*2.5))
  for (j in 1:p) {
    beta[j] ~ dnorm(0, 1/(2.5*2.5))
  }
  for (r in 2:R) {
    gamma[r] ~ dnorm(0, 1/(2.5*2.5))
  }
  gamma[1] <- 0

  phi ~ dgamma(2, 0.1)
}
"

cat("✅ Model 3 (Hierarchical Beta-Binomial) defined\n")

# TODO: Compile & run JAGS model
# jags_data3 <- c(jags_data, list(country = ..., C = ...))
# model3 <- jags.model(textConnection(model3_string), data = jags_data3,
#                       n.chains = mcmc_cfg$n_chains, n.adapt = 1000)
# update(model3, mcmc_cfg$n_burnin)
# samples3 <- coda.samples(model3,
#   variable.names = c("beta0", "beta", "gamma", "phi", "sigma_u"),
#   n.iter = mcmc_cfg$n_iter, thin = mcmc_cfg$n_thin)

---
## Section E — MCMC Diagnostics

In [ ]:
# E0) MCMC Diagnostics (per PROJECT_PLAN.md §11)
#
# For each model:
# 1. Trace plots
# 2. Posterior density plots
# 3. Autocorrelation plots
# 4. R-hat (Gelman-Rubin)
# 5. Effective sample size (ESS)
# 6. Geweke diagnostics

# Example (uncomment after fitting models):
# summary(samples1)
# gelman.diag(samples1)
# effectiveSize(samples1)
# traceplot(samples1)

cat("⏳ MCMC diagnostics — run after fitting models\n")

---
## Section F — Posterior Inference & Model Comparison

In [ ]:
# F0) Posterior Inference (per PROJECT_PLAN.md §13)
#
# For each model:
# 1. Posterior means & medians
# 2. 95% equal-tail credible intervals
# 3. 95% HPD intervals
# 4. Posterior probabilities of directional effects

# Example:
# MCMCsummary(samples1, round = 4)
# HPDinterval(samples1)

cat("⏳ Posterior inference — implement after fitting models\n")

In [ ]:
# F1) Posterior Predictive Checks (per PROJECT_PLAN.md §14)
#
# Test quantities:
# T1: Mean treatment success rate
# T2: Variance of success rates (key diagnostic)
# T3: Count of low-success country-years
# T4: Within-region variance

cat("⏳ Posterior predictive checks — implement after fitting models\n")

In [ ]:
# F2) Model Comparison via DIC (per PROJECT_PLAN.md §15)
#
# IMPORTANT: Compute observed-data DIC (beta-binomial log-PMF)
# for Models 2 & 3, not JAGS's default conditional DIC.

# Example:
# dic1 <- dic.samples(model1, n.iter = 5000, type = "pD")
# For Models 2 & 3: compute manually from posterior draws

cat("⏳ DIC model comparison — implement after fitting models\n")

---
## Section G — Parameter Recovery Simulation

In [ ]:
# G0) Parameter Recovery (per PROJECT_PLAN.md §12)
#
# For each model:
# 1. Choose true parameter values (from posteriors)
# 2. Simulate 50 datasets (same design structure)
# 3. Refit model to each synthetic dataset
# 4. Evaluate: bias, RMSE, coverage

cat("⏳ Parameter recovery simulation — implement after main analysis\n")

---
## Section H — Frequentist Comparison (Bonus)

In [ ]:
# H0) Frequentist Comparison (per PROJECT_PLAN.md §16)
#
# M1 freq: glm(cbind(success, cohort - success) ~ ..., family = binomial)
# M2 freq: VGAM::vglm(..., family = betabinomial)
# M3 freq: lme4::glmer(cbind(success, cohort - success) ~ ... + (1|country), family = binomial)

cat("⏳ Frequentist comparison — implement after main Bayesian analysis\n")

---
## Section I — Sensitivity Analyses

In [ ]:
# I0) Sensitivity Analyses (per PROJECT_PLAN.md §17)
#
# 1. cohort ≥ 50 vs. cohort > 0
# 2. Main predictors vs. + e_tbhiv_prct
# 3. phi prior: Gamma(2,0.1) vs. Gamma(1,0.1)
# 4. sigma_u prior: HalfNormal(0,1) vs. HalfNormal(0,2.5)
# 5. Post-2021 definitional check (used_2021_defs_flg == 1)

cat("⏳ Sensitivity analyses — implement after main results confirmed\n")

---
## Section J — Save Results

In [ ]:
# J0) Save all outputs to Drive

cat("\n" , paste(rep("=", 60), collapse = ""), "\n")
cat("  RUN COMPLETE\n")
cat(paste(rep("=", 60), collapse = ""), "\n\n")
cat("📁 All outputs saved to Drive:\n")
cat("   ", RUN_DIR, "\n")
cat("\n📋 Session Info:\n")
sessionInfo()